# Advanced 03 — Governance and Production Readiness

Admit authenticated, candidate-bound evidence; distinguish failed controls from missing assurance; and convert a signed readiness decision into one narrow deployment authorization.

![Evidence-bound production release](architecture.svg)

The model is outside the authority path. Evidence producers assert typed claims; the release gate applies current policy; an independent approver authorizes one exact deployment.

## 1. Load the course lab

The notebook imports the reusable course module rather than copying its security logic.

In [ ]:
import runpy
from datetime import datetime, timedelta, timezone
from dataclasses import replace
ns = runpy.run_path('03_production_gate.py')
build_scenario = ns['build_scenario']
attest_evidence, attest_risk = ns['attest_evidence'], ns['attest_risk']
artifact_for = ns['artifact_for']
DecisionState, DeploymentLedger = ns['DecisionState'], ns['DeploymentLedger']
DEMO_PRODUCER_KEYS, DEMO_HUMAN_KEYS = ns['DEMO_PRODUCER_KEYS'], ns['DEMO_HUMAN_KEYS']
now = datetime(2026,9,20,tzinfo=timezone.utc)
gate, candidate, evidence, risks, approver = build_scenario(now=now)

## 2. Establish the safe baseline

Observe the trusted inputs and the decision evidence before injecting failures.

In [ ]:
decision = gate.evaluate(candidate,evidence,risks,now=now)
assert decision.state is DecisionState.READY and gate.verify_decision(decision)
{'state': decision.state.value, 'decision_id': decision.decision_id, 'evidence_ids': decision.evidence_ids, 'risk_acceptances': decision.risk_acceptance_ids}

## 3. Inject an attack

Change one security-relevant boundary and keep the rest of the fixture stable.

In [ ]:
attacked = list(evidence)
attack_index = next(i for i,item in enumerate(attacked) if artifact_for(item.payload).kind == 'attack-evaluation')
attack_payload = replace(attacked[attack_index].payload,severe_attack_successes=1)
attacked[attack_index] = attest_evidence(attack_payload,DEMO_PRODUCER_KEYS[attack_payload.artifact.producer])
severe = gate.evaluate(candidate,attacked,risks,now=now)
assert severe.state is DecisionState.BLOCKED and 'blocked:severe-attack-successes:1' in severe.blockers
severe

## 4. Attempt a bypass

The assertions below make the security property executable and regression-testable.

In [ ]:
tampered = list(evidence)
tampered[0] = replace(tampered[0],payload=replace(tampered[0].payload,artifact=replace(tampered[0].payload.artifact,owner='attacker')))
tampered_decision = gate.evaluate(candidate,tampered,risks,now=now)
assert tampered_decision.state is DecisionState.INCOMPLETE
assert any(item.startswith('incomplete:invalid-attestation:') for item in tampered_decision.blockers)
tampered_decision.blockers

## 5. Evaluate observable outcomes

Use explicit denominators or counts. Private model reasoning is neither required nor recorded.

In [ ]:
{'state': decision.state.value, 'admitted_evidence': len(decision.evidence_ids), 'required_evidence': len(gate.policy.requirements), 'blocker_count': len(decision.blockers), 'accepted_risks': len(decision.risk_acceptance_ids)}

## 6. Exercise a second failure mode

In [ ]:
expired_payload = replace(risks[0].payload,expires_at=now)
expired = attest_risk(expired_payload,DEMO_HUMAN_KEYS[expired_payload.approver])
expired_decision = gate.evaluate(candidate,evidence,[expired],now=now)
assert expired_decision.state is DecisionState.INCOMPLETE
expired_decision.blockers

## 7. Independent, exact, single-use deployment authorization

A READY receipt is reviewed by a current independent approver. The deployer rechecks exact subject and policy bindings, then consumes the authorization atomically.

In [ ]:
authorization_result = gate.authorize_deployment(decision,candidate,approver,authorization_id='DA-notebook',now=now+timedelta(minutes=1))
assert authorization_result.allowed
authorization = authorization_result.authorization
ledger = DeploymentLedger(gate)
changed_candidate = replace(candidate,artifact_digest=ns['_digest']('changed-candidate'))
wrong_target = ledger.consume(authorization,changed_candidate,current_policy_version=gate.policy.version,now=now+timedelta(minutes=2))
first = ledger.consume(authorization,candidate,current_policy_version=gate.policy.version,now=now+timedelta(minutes=2))
replay = ledger.consume(authorization,candidate,current_policy_version=gate.policy.version,now=now+timedelta(minutes=2))
assert wrong_target.reason == 'authorization-binding' and first.allowed and replay.reason == 'authorization-replayed'
{'wrong_target': wrong_target.reason, 'first': first.reason, 'replay': replay.reason}

## 8. OpenTelemetry SDK decision trace

The adapter uses the pinned OpenTelemetry Python SDK and an in-memory exporter. Only allowlisted decision metadata is recorded; the SDK never becomes the policy authority.

In [ ]:
otel = runpy.run_path('03_production_gate_otel.py')
otel_decision, spans = otel['credential_free_demo']()
attributes = dict(spans[0].attributes)
assert otel_decision.ready and len(spans) == 1
assert 'evidence.example' not in repr(attributes) and 'operator fallback' not in repr(attributes)
attributes

## 9. Decision-state semantics

FAILED is trustworthy negative evidence. ERROR or missing execution means the assurance claim is incomplete. Preserve both meanings instead of flattening them into one score.

In [ ]:
coverage = list(evidence)
idx = next(i for i,item in enumerate(coverage) if artifact_for(item.payload).kind == 'attack-evaluation')
partial_payload = replace(coverage[idx].payload,executed_attempts=23)
coverage[idx] = attest_evidence(partial_payload,DEMO_PRODUCER_KEYS[partial_payload.artifact.producer])
partial = gate.evaluate(candidate,coverage,risks,now=now)
assert partial.state is DecisionState.INCOMPLETE and 'incomplete:attack-coverage:23/24' in partial.blockers
partial.blockers

## 10. Production replacement

Production replacement: workload identity and asymmetric/verifiable attestations; signed policy distribution; durable decision and single-use authorization state; enterprise identity with revocation and separation of duties; protected evidence retention; environment protection; staged rollout; unknown-outcome reconciliation; rollback and kill switches; privacy-controlled telemetry; and event-driven reevaluation after any material change. The local HMAC keys and lock prove invariants inside one process, not distributed trust or consistency.

## 11. Exercises

1. Add a signed model-card evidence type bound to the model digest.
2. Implement a non-production policy and prove its decision cannot authorize production.
3. Replace HMAC evidence with a locally verifiable asymmetric signature.
4. Add durable compare-and-swap for authorization consumption.
5. Model an UNKNOWN deployment outcome and reconcile it before retry.
6. Build an OPA, Cedar, GitHub deployment-protection, or Sigstore adapter that preserves the same trusted boundary.

## Checkpoint

Explain which trusted component enforces the invariant, what evidence proves the decision, and what residual risk remains.